# Build Metadata Map

In [ ]:
import json
from pathlib import Path

meta_path = Path("data/meta_2014.json")

with open(meta_path) as f:
    meta = json.load(f)

id_to_type = {}
for entry in meta:
    idx = int(entry["id"])
    typ = entry["meta"]["appliance"]["type"]
    id_to_type[idx] = typ

print("Unique appliance types:")
print(sorted(set(id_to_type.values())))


Unique appliance types:
['Air Conditioner', 'Compact Fluorescent Lamp', 'Fan', 'Fridge', 'Hairdryer', 'Heater', 'Incandescent Light Bulb', 'Laptop', 'Microwave', 'Vacuum', 'Washing Machine']


# Convert Every Recording to Power Series

In [ ]:
import numpy as np
import pandas as pd
from plaid_adapter import waveform_to_power_series

data_dir = Path("data/2014/2014")

power_series_by_type = {}

for idx, typ in id_to_type.items():

    file = data_dir / f"{idx}.csv"
    if not file.exists():
        print(f"[ERROR] File not found: {file}")
        continue

    df = pd.read_csv(file, header=None, names=['current', 'voltage'])

    voltage = df["voltage"].values
    current = df["current"].values

    power_series = waveform_to_power_series(voltage, current)

    power_series_by_type.setdefault(typ, []).append(power_series)


# Extract Per-Recording Statistics

In [ ]:
def extract_behavior_stats(power):
    power = np.array(power)

    steady = power[int(len(power)*0.6):]  # ignore transient part
    baseline = np.mean(steady)

    peak = np.max(power)
    spike_ratio = peak / baseline if baseline > 1 else 1

    # settling time: when signal enters steady band
    band = baseline * 0.1
    settling_idx = np.argmax(np.abs(power - baseline) < band)

    return {
        "mean_power": baseline,
        "std_steady": np.std(steady),
        "spike_ratio": spike_ratio,
        "settling_time": settling_idx,
        "variance": np.var(power)
    }


# Aggregate Per Appliance Type

In [ ]:
stats_by_type = {}

for typ, series_list in power_series_by_type.items():

    records = [extract_behavior_stats(s) for s in series_list]

    stats_by_type[typ] = pd.DataFrame(records)


In [21]:
stats_by_type["Fan"].describe()

,mean_power,std_steady,spike_ratio,settling_time,variance
count,115.000000,115.000000,115.000000,115.000000,115.000000
mean,61.559624,0.686016,1.056942,5.191304,484.697433
std,27.185249,0.960580,0.094880,4.043395,481.375744
min,16.299623,0.019481,1.000719,2.000000,21.500420
25%,42.174608,0.145155,1.010191,3.000000,124.568498
50%,56.666885,0.277747,1.023437,4.000000,380.359624
75%,78.409645,0.785453,1.053422,5.000000,647.954507
max,114.892295,4.362663,1.380631,37.000000,2267.989871


In [23]:

stats_by_type["Incandescent Light Bulb"].describe()

,mean_power,std_steady,spike_ratio,settling_time,variance
count,114.000000,114.000000,114.000000,114.000000,114.000000
mean,50.539252,0.388586,1.289336,4.464912,313.336099
std,21.191768,0.797900,0.361338,2.681235,388.715112
min,24.445415,0.018447,1.000981,0.000000,8.636662
25%,30.576965,0.034525,1.067840,3.000000,82.955875
50%,44.402372,0.045906,1.228033,4.000000,225.717848
75%,65.723708,0.077561,1.425590,5.000000,368.866928
max,112.146357,3.179338,4.413996,21.000000,3371.903546
